# Feature Engineering na Prática
## Problema Modelo: Previsão de Preço de Imóveis

Neste notebook construímos progressivamente um pipeline de Feature Engineering
usando um dataset sintético de imóveis. A cada seção aplicamos uma técnica nova
e medimos seu impacto em um modelo de regressão Ridge via validação cruzada.

### Estrutura

```
1. Importações e configuração
2. Geração dos dados e análise exploratória
3. Funções de pré-processamento (uma por técnica)
4. Funções de modelagem e avaliação
5. Experimentos
   5.0  Linha de base (baseline)
   5.1  Interações
   5.2  Polinômios
   5.3  Agregações e data leakage
   5.4  Transformações (log, sqrt, Box-Cox)
   5.5  Domínio específico
   5.6  Scaling (StandardScaler, MinMaxScaler, RobustScaler)
   5.7  Feature Learning (PCA como extrator)
6. Resultados consolidados
7. Conclusão e regras práticas
```

| Técnica | O que fazemos |
|---------|---------------|
| **Interações** | `area × quartos`, `area × dist_centro` |
| **Polinômios** | `area²`, `area³`, `dist²` |
| **Agregações** | Média de preço por bairro via Target Encoding OOF |
| **Transformações** | `log(area)`, `sqrt(distancia)`, Box-Cox |
| **Domínio** | Índice de valorização, área por quarto, flag premium |
| **Scaling** | StandardScaler vs MinMaxScaler vs RobustScaler |
| **Feature Learning** | PCA como extrator de componentes latentes |


## 1. Importações e Configuração


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
from itertools import combinations
from scipy import stats
warnings.filterwarnings('ignore')

from sklearn.linear_model    import Ridge, RidgeCV
from sklearn.preprocessing   import (PolynomialFeatures, StandardScaler,
                                      MinMaxScaler, RobustScaler)
from sklearn.model_selection import cross_val_score, KFold, train_test_split
from sklearn.pipeline        import Pipeline
from sklearn.decomposition   import PCA
from sklearn.metrics         import r2_score

SEED = 42
np.random.seed(SEED)
plt.style.use('seaborn-v0_8-whitegrid')

# Paleta de cores
C_BASE   = '#5A9BD4'   # baseline / polinomios
C_GOOD   = '#2EAA6E'   # resultados positivos / agregacoes
C_WARN   = '#E07B39'   # atencao / dominio especifico
C_BAD    = '#D94F4F'   # resultados negativos / leakage
C_PURPLE = '#7E6CB2'   # interacoes / complexidade estrutural
C_TEAL   = '#0F9ED5'   # scaling
C_GRAY   = '#7F8C8D'   # feature learning

BASE_COLS = ['area', 'quartos', 'banheiros', 'idade', 'dist_centro', 'vagas']

print('Importacoes concluidas.')


## 2. Geração dos Dados e Análise Exploratória

### Contexto do problema

O dataset simula o **registro de 800 imóveis residenciais** com suas
características físicas e de localização. O objetivo é prever o `preco`
com base nessas variáveis — um problema típico de avaliação imobiliária
(*automated valuation model*).

### Variáveis do dataset

| Variável | Tipo | Descrição |
|----------|------|-----------|
| `bairro` | Categórica | 5 bairros com multiplicadores de valorização distintos |
| `area` | Numérica | Área útil em m² (distribuição log-normal, 30–450 m²) |
| `quartos` | Inteira | Número de quartos (1–5) |
| `banheiros` | Inteira | Número de banheiros (correlacionado com quartos) |
| `idade` | Inteira | Idade do imóvel em anos (0–50) |
| `dist_centro` | Numérica | Distância ao centro em km (distribuição exponencial) |
| `vagas` | Inteira | Vagas de garagem (0–3) |
| `preco` | Alvo | Preço em R$ (limitado a 50k–5M) |

### Como o preço é gerado

O alvo contém relações **não-lineares intencionais** para que cada técnica
de feature engineering tenha um sinal real para capturar:

```
preco = (3000 × area
       + 800 × area × quartos       ← interacao real
       - 15  × area × dist_centro   ← interacao negativa real
       + 0.8 × area²                ← termo quadratico real
       + 18000 × quartos
       - 1200  × idade
       + 15000 × vagas)
       × mult_bairro
       × exp(ruido)
```

Em todos os modelos utilizamos `log1p(preco)` como alvo para estabilizar
a variância.


In [ ]:
N = 800

BAIRROS     = ['Centro', 'Norte', 'Sul', 'Leste', 'Oeste']
BAIRRO_MULT = {'Centro': 1.6, 'Norte': 1.1, 'Sul': 0.85, 'Leste': 1.0, 'Oeste': 0.9}

bairro_col  = np.random.choice(BAIRROS, N)
area        = np.random.lognormal(mean=5.0, sigma=0.45, size=N).clip(30, 450)
quartos     = np.random.choice([1,2,3,4,5], N, p=[0.10,0.25,0.35,0.20,0.10])
banheiros   = np.clip(quartos - np.random.choice([0,1], N, p=[0.6,0.4]), 1, 5)
idade       = np.random.exponential(scale=12, size=N).clip(0, 50).astype(int)
dist_centro = np.random.exponential(scale=8,  size=N).clip(0.5, 40)
vagas       = np.random.choice([0,1,2,3], N, p=[0.15,0.40,0.35,0.10])

mult  = np.array([BAIRRO_MULT[b] for b in bairro_col])
noise = np.random.normal(0, 0.08, N)

preco = (
    3000  * area
    + 800 * area * quartos
    - 15  * area * dist_centro
    + 0.8 * area**2
    + 18000 * quartos
    - 1200  * idade
    + 15000 * vagas
) * mult * np.exp(noise)

df = pd.DataFrame({
    'bairro'     : bairro_col,
    'area'       : area.round(1),
    'quartos'    : quartos,
    'banheiros'  : banheiros,
    'idade'      : idade,
    'dist_centro': dist_centro.round(2),
    'vagas'      : vagas,
    'preco'      : preco.clip(50_000, 5_000_000).round(-2),
})

y = np.log1p(df['preco'])

print(f'Dataset: {df.shape[0]} imoveis x {df.shape[1]} colunas')
print(f'Preco mediano: R$ {df["preco"].median():,.0f}')
print(f'Preco min/max: R$ {df["preco"].min():,.0f} / R$ {df["preco"].max():,.0f}')
df.head(6)


In [ ]:
# EDA: distribuicoes e relacoes com o target
# Grade superior: histogramas das 6 features + distribuicao do alvo
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
fig.suptitle('EDA — Distribuicoes e relacao com log(preco)', fontsize=13, fontweight='bold')

for ax, col in zip(axes[0], ['area', 'quartos', 'banheiros', 'idade']):
    ax.hist(df[col], bins=28, color=C_BASE, edgecolor='white', alpha=0.85)
    ax.set_title(col, fontweight='bold')

for ax, col in zip(axes[1], ['dist_centro', 'vagas', 'preco', 'bairro']):
    if col == 'preco':
        ax.hist(y, bins=28, color=C_GOOD, edgecolor='white', alpha=0.85)
        ax.set_title('log1p(preco) — alvo', fontweight='bold')
    elif col == 'bairro':
        counts = df['bairro'].value_counts()
        ax.bar(counts.index, counts.values, color=C_PURPLE, edgecolor='white', alpha=0.85)
        ax.set_title('bairro (contagem)', fontweight='bold')
        ax.tick_params(axis='x', rotation=30)
    else:
        ax.hist(df[col], bins=20, color=C_WARN, edgecolor='white', alpha=0.85)
        ax.set_title(col, fontweight='bold')

plt.tight_layout()
plt.show()

# Grade: scatter de cada feature numerica vs log(preco)
# Permite visualizar diretamente o tipo de relacao (linear, curva, negativa)
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.suptitle('Relacao de cada feature com log(preco)', fontsize=13, fontweight='bold')

scatter_vars = ['area', 'quartos', 'banheiros', 'idade', 'dist_centro', 'vagas']
colors_sc    = [C_PURPLE, C_BASE, C_BASE, C_WARN, C_BAD, C_GOOD]

for ax, col, color in zip(axes.flat, scatter_vars, colors_sc):
    ax.scatter(df[col], y, alpha=0.25, s=14, color=color)
    # Adicionar linha de tendencia linear simples
    z = np.polyfit(df[col], y, 1)
    x_line = np.linspace(df[col].min(), df[col].max(), 100)
    ax.plot(x_line, np.polyval(z, x_line), color='black', lw=1.5, ls='--', alpha=0.6)
    corr_val = df[col].corr(y)
    ax.set_xlabel(col)
    ax.set_ylabel('log1p(preco)')
    ax.set_title(f'{col}  (rho = {corr_val:.3f})', fontweight='bold')

plt.tight_layout()
plt.show()

# Grafico de correlacoes — gerado DEPOIS dos scatters para que o texto
# de observacoes reflita os valores reais calculados
corrs = df[BASE_COLS].corrwith(y).sort_values()
fig, ax = plt.subplots(figsize=(7, 3.5))
colors_c = [C_GOOD if v > 0 else C_BAD for v in corrs]
ax.barh(corrs.index, corrs.values, color=colors_c, edgecolor='white')
ax.axvline(0, color='gray', lw=0.8)
ax.set_xlabel('Correlacao de Pearson com log(preco)')
ax.set_title('Correlacao das features brutas com o alvo', fontweight='bold')
plt.tight_layout()
plt.show()

# Observacoes baseadas nos valores calculados acima
print('Observacoes:')
for col, val in corrs.items():
    direcao = 'positiva' if val > 0 else 'negativa'
    print(f'  - {col:<12}: rho = {val:+.3f}  ({direcao})')
print()
print('Destaque: area tem a correlacao mais alta mas relacao claramente curva')
print('  (veja o scatter) — polinomios e interacoes capturarao esse sinal.')


## 3. Funções de Pré-processamento

Cada função recebe o DataFrame original e devolve `X` pronto para modelagem.
**Nenhum experimento é executado nesta seção** — apenas as transformações
são declaradas.

Regras de implementação:
- As funções não chamam `display()` nem `print()` internamente.
- Nenhuma função modifica o DataFrame original (operam em cópias).
- As features da etapa anterior são sempre passadas como parâmetro,
  garantindo que cada experimento acumule sobre a etapa anterior.


In [ ]:
# 3.0 Baseline
def build_baseline(df_in):
    """Features numericas brutas — ponto de partida para todos os experimentos."""
    return df_in[BASE_COLS].copy()


# ─────────────────────────────────────────────────────────────────────
# 3.1 Interacoes
#
# Passos:
#   1. Identificar pares de variaveis com relacao conjunta ao target.
#   2. Criar nova coluna = produto das duas variaveis (x1 * x2).
#   3. Adicionar ao conjunto de features existente.
#
# Resultado: o modelo linear pode agora capturar o efeito combinado.
# Atencao: com n features, existem C(n,2) pares possiveis — prefira
# pares motivados pelo dominio a gerar todas as combinacoes.
# ─────────────────────────────────────────────────────────────────────
def build_interacoes(df_in, X_base):
    """
    Combinacoes multiplicativas de variaveis para capturar efeitos conjuntos.
    Pares escolhidos com base no dominio imobiliario:
      - area x quartos    : imoveis maiores com mais quartos valem muito mais
      - area x banheiros  : mesma logica de qualidade
      - area x dist_centro: penalizacao por localizacao proporcional ao tamanho
      - quartos x vagas   : combinacao de conforto interno e externo
    """
    # Passo 1-2: criar cada interacao como produto direto
    X = X_base.copy()
    X['area_x_quartos']   = df_in['area'] * df_in['quartos']
    X['area_x_banheiros'] = df_in['area'] * df_in['banheiros']
    X['area_x_dist']      = df_in['area'] * df_in['dist_centro']
    X['quartos_x_vagas']  = df_in['quartos'] * df_in['vagas']
    # Passo 3: retornar o DataFrame ampliado
    return X


# ─────────────────────────────────────────────────────────────────────
# 3.2 Polinomios
#
# Passos:
#   1. Selecionar variaveis com relacao curva com o target.
#   2. Elevar ao quadrado (x**2) e ao cubo (x**3) quando justificado.
#   3. Adicionar ao conjunto existente.
#
# Resultado: o modelo linear consegue ajustar curvas.
# Atencao: PolynomialFeatures(degree=2) em todas as colunas cria
# n*(n+3)/2 features — crescimento quadratico, risco de overfitting.
# ─────────────────────────────────────────────────────────────────────
def build_polinomios(df_in, X_int):
    """
    Termos quadraticos e cubicos para variaveis com relacao curva com o preco.
    area² / area³ : preco cresce mais que linearmente com a area
    dist²         : penalizacao por distancia nao e linear
    idade²        : desvalorizacao acelera em imoveis muito antigos
    """
    X = X_int.copy()
    # Passo 1-2: elevar ao quadrado e ao cubo as variaveis selecionadas
    X['area2']  = df_in['area'] ** 2
    X['area3']  = df_in['area'] ** 3
    X['dist2']  = df_in['dist_centro'] ** 2
    X['idade2'] = df_in['idade'] ** 2
    return X


# ─────────────────────────────────────────────────────────────────────
# 3.3 Agregacoes (Target Encoding OOF)
#
# Passos:
#   1. Dividir os dados em n_splits folds (KFold).
#   2. Para cada fold, calcular a media do target SOMENTE no fold de treino.
#   3. Aplicar suavizacao (smoothing) para estabilizar grupos pequenos.
#   4. Atribuir o valor suavizado as amostras do fold de validacao.
#
# Resultado: cada imovel recebe a media do preco do seu bairro,
# calculada sem "ver" o proprio target (sem leakage).
# ─────────────────────────────────────────────────────────────────────
def target_encode_oof(df_in, y_series, col, smoothing=10, n_splits=5):
    """
    Target encoding OOF: media do target calculada somente nos folds de treino.
    Formula de suavizacao: enc(j) = (n_j*media_j + m*media_global) / (n_j + m)
    """
    global_mean = y_series.mean()
    encoded     = np.zeros(len(df_in))
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=SEED)
    for tr_idx, va_idx in kf.split(df_in):
        tr_y   = y_series.iloc[tr_idx]
        tr_col = df_in[col].iloc[tr_idx]
        va_col = df_in[col].iloc[va_idx]
        # Passo 2: estatisticas somente do fold de treino
        agg = tr_y.groupby(tr_col).agg(['mean', 'count'])
        # Passo 3: suavizacao
        agg['smooth'] = ((agg['mean'] * agg['count'] + global_mean * smoothing)
                         / (agg['count'] + smoothing))
        # Passo 4: atribuir ao fold de validacao
        encoded[va_idx] = va_col.map(agg['smooth']).fillna(global_mean).values
    return encoded


def build_agregacoes(df_in, y_series, X_poly):
    """
    Adiciona estatisticas por bairro:
      te_bairro         : media do log(preco) por bairro (OOF — sem leakage)
      media_area_bairro : area media dos imoveis no bairro (sem target)
      count_bairro      : numero de imoveis no bairro (proxy de infraestrutura)
    """
    X = X_poly.copy()
    y_s = pd.Series(y_series.values, index=df_in.index)
    X['te_bairro']         = target_encode_oof(df_in.reset_index(drop=True),
                                               y_s.reset_index(drop=True), 'bairro')
    X['media_area_bairro'] = df_in.groupby('bairro')['area'].transform('mean').values
    X['count_bairro']      = df_in.groupby('bairro')['area'].transform('count').values
    return X


# ─────────────────────────────────────────────────────────────────────
# 3.4 Transformacoes
#
# Passos:
#   1. Identificar variaveis com distribuicao assimetrica (skew alto).
#   2. Aplicar log1p, sqrt ou Box-Cox para aproximar de uma distribuicao
#      mais simetrica.
#   3. Adicionar a coluna transformada ao conjunto existente (manter original).
#
# Resultado: o Ridge consegue atribuir coeficientes mais estaveis
# quando as features nao tem caudas longas extremas.
# ─────────────────────────────────────────────────────────────────────
def build_transformacoes(df_in, X_agg):
    """
    Transformacoes matematicas para corrigir distribuicoes assimetricas:
      log1p(area)        : area tem distribuicao lognormal
      sqrt(dist_centro)  : distancia tem cauda longa
      log1p(idade)       : concentracao em imoveis novos + cauda longa
      sqrt(area)         : alternativa suave ao log
      boxcox(area)       : transformacao otima encontrada automaticamente
    """
    X = X_agg.copy()
    # Passos 1-2: aplicar cada transformacao
    X['log_area']         = np.log1p(df_in['area'])           # Passo 2a
    X['sqrt_dist_centro'] = np.sqrt(df_in['dist_centro'])     # Passo 2b
    X['log_idade']        = np.log1p(df_in['idade'])          # Passo 2c
    X['sqrt_area']        = np.sqrt(df_in['area'])            # Passo 2d
    area_bc, lmbda        = stats.boxcox(df_in['area'] + 1)   # Passo 2e: lambda otimo
    X['boxcox_area']      = area_bc
    return X, lmbda


# ─────────────────────────────────────────────────────────────────────
# 3.5 Dominio Especifico
#
# Passos:
#   1. Consultar conhecimento do dominio (mercado imobiliario).
#   2. Criar features que traduzem regras que um especialista conhece.
#   3. Validar com correlacao: a nova feature tem sinal com o target?
#
# Resultado: features que capturam logica de negocio que dados brutos
# nao expressam diretamente.
# ─────────────────────────────────────────────────────────────────────
def build_dominio(df_in, X_transf):
    """
    Features criadas com conhecimento do mercado imobiliario:
      area_por_quarto   : conforto interno — area media disponivel por quarto
      indice_comodidade : score composto de quartos, banheiros e vagas
      penalidade_idade  : desconto de ~1.2% ao ano apos 10 anos de uso
      is_premium        : flag para imoveis a menos de 5 km do centro
      dist_premium      : distancia ate o limiar de 5 km (zero se distante)
      ratio_ban_quarto  : proporcao banheiros/quartos (indicador de luxo)
      potencial_aluguel : indice de retorno estimado
    """
    X = X_transf.copy()
    # Passo 2: criar cada feature de dominio
    X['area_por_quarto']   = df_in['area'] / df_in['quartos']
    X['indice_comodidade'] = (df_in['quartos']
                               + df_in['banheiros'] * 0.8
                               + df_in['vagas'] * 0.5)
    X['penalidade_idade']  = np.where(df_in['idade'] > 10,
                                       (df_in['idade'] - 10) * 0.012, 0)
    X['is_premium']        = (df_in['dist_centro'] < 5).astype(int)
    X['dist_premium']      = np.where(df_in['dist_centro'] < 5,
                                       5 - df_in['dist_centro'], 0)
    X['ratio_ban_quarto']  = df_in['banheiros'] / df_in['quartos']
    X['potencial_aluguel'] = ((df_in['quartos'] * 0.4 + df_in['vagas'] * 0.2)
                               / np.log1p(df_in['dist_centro']))
    return X


print('Funcoes de pre-processamento definidas.')


## 4. Função de Modelagem e Avaliação

Todas as comparações utilizam o mesmo protocolo:
- **Ridge com StandardScaler** — sensível à escala das features, ideal para
  expor o valor incremental de cada transformação.
- **Métrica:** R² médio em validação cruzada com 5 partições (KFold).
- **Os resultados são acumulados em `RESULTADOS`** para o comparativo final.


In [ ]:
RESULTADOS = {}

def avaliar(X, y, label, alpha=10.0):
    """
    Avalia um conjunto de features com Ridge + CV 5-fold.

    Parametros
    ----------
    X     : DataFrame de features
    y     : Series com o target (log1p do preco)
    label : nome do experimento — chave em RESULTADOS
    alpha : regularizacao do Ridge

    Retorna
    -------
    r2_mean : R² medio dos 5 folds
    """
    pipe = Pipeline([('scaler', StandardScaler()), ('ridge', Ridge(alpha=alpha))])
    cv   = KFold(n_splits=5, shuffle=True, random_state=SEED)
    r2s  = cross_val_score(pipe, X, y, cv=cv, scoring='r2')
    r2_mean = float(r2s.mean())
    RESULTADOS[label] = {'r2': r2_mean, 'std': float(r2s.std()),
                          'n_features': X.shape[1]}
    return r2_mean

print('Funcao avaliar definida.')


## 5. Experimentos


### 5.0 Linha de Base

**Objetivo:** estabelecer o ponto de referência antes de qualquer
engenharia de features. O modelo recebe somente as seis variáveis
numéricas brutas.

**Racional:** o R² de linha de base mede quanto da variância do preço
já está capturado pelas features originais. Qualquer técnica que não
supere esse valor não agrega nada ao modelo.


In [ ]:
print('=' * 60)
print('BASELINE — features numericas brutas, sem transformacoes')
print('=' * 60)

X_base = build_baseline(df)
r2_base = avaliar(X_base, y, 'Baseline')
print(f'  R² = {r2_base:.4f}  ({X_base.shape[1]} features)')
print(f'  Features: {X_base.columns.tolist()}')


**Conclusão:** as features brutas capturam uma parcela relevante da variância
do preço. A distância ao experimento seguinte medirá o ganho real de cada
técnica de engenharia de features.


### 5.1 Interações

**Objetivo:** demonstrar que o produto de duas variáveis pode capturar
efeitos que nenhum coeficiente linear individual consegue expressar.

**Racional:** o modelo linear aprende `β₁ × area + β₂ × quartos`, mas não
consegue representar "imóveis grandes com muitos quartos valem
desproporcionalmente mais". Para isso é necessário criar explicitamente a
coluna `area × quartos` — só então o Ridge aprende um coeficiente para
esse efeito conjunto.

**Risco:** com *n* features existem *C(n,2)* pares possíveis. Gerar todos
por força bruta cria ruído e retorno decrescente. Interações motivadas
pelo domínio têm muito mais sinal do que combinações aleatórias.


In [ ]:
print('=' * 60)
print('INTERACOES — combinacoes multiplicativas de variaveis')
print('=' * 60)

# ── Por que este experimento usa um mini-dataset ──────────────────────
# No dataset de imoveis, a feature area ja e tao dominante (rho=0.77)
# que area*quartos e quase colinear com area. O Ridge, com regularizacao
# L2, suprime termos redundantes e a interacao parece nao agregar.
#
# Para demonstrar o efeito de forma clara e inequivoca, usamos um
# mini-dataset sintetico onde a interacao e EXATAMENTE o sinal —
# sem area mascando o resultado.
#
# Depois voltamos ao dataset real para mostrar o contraste:
# features informativas (mesmo com efeito menor) vs colunas de ruido.

# ── Parte 1: mini-dataset — interacao como sinal puro ────────────────
np.random.seed(SEED)
N2 = 500
x1 = np.random.uniform(1, 10, N2)
x2 = np.random.uniform(1, 10, N2)
# Efeitos individuais fracos, interacao forte — Ridge precisa de x1*x2
y_mini = pd.Series(0.2*x1 + 0.2*x2 + 5*x1*x2 + np.random.normal(0, 3, N2))

sets_mini = {
    'Apenas x1 e x2':          pd.DataFrame({'x1': x1, 'x2': x2}),
    '+ interacao real x1*x2':   pd.DataFrame({'x1': x1, 'x2': x2, 'x1_x_x2': x1*x2}),
    '+ 8 colunas de ruido':     pd.DataFrame({'x1': x1, 'x2': x2,
                                               **{f'ruido_{k}': np.random.permutation(x1*x2)
                                                  for k in range(8)}}),
}

kf5 = KFold(n_splits=5, shuffle=True, random_state=SEED)
r2_mini = {}
for label, X_d in sets_mini.items():
    pipe = Pipeline([('sc', StandardScaler()), ('r', Ridge(alpha=1.0))])
    r2s  = cross_val_score(pipe, X_d, y_mini, cv=kf5, scoring='r2')
    r2_mini[label] = r2s.mean()

# Visualizacao 1: por que a interacao importa (mini-dataset)
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('Interacao como sinal: y = 0.2*x1 + 0.2*x2 + 5*x1*x2 + ruido',
             fontweight='bold')

axes[0].scatter(x1, y_mini, alpha=0.3, s=14, color=C_BASE)
axes[0].set_xlabel('x1'); axes[0].set_ylabel('y')
axes[0].set_title(f'x1 vs y  (rho={np.corrcoef(x1,y_mini)[0,1]:.3f})')

axes[1].scatter(x2, y_mini, alpha=0.3, s=14, color=C_BASE)
axes[1].set_xlabel('x2'); axes[1].set_ylabel('y')
axes[1].set_title(f'x2 vs y  (rho={np.corrcoef(x2,y_mini)[0,1]:.3f})')

axes[2].scatter(x1*x2, y_mini, alpha=0.3, s=14, color=C_GOOD)
axes[2].set_xlabel('x1 * x2'); axes[2].set_ylabel('y')
axes[2].set_title(f'interacao x1*x2 vs y  (rho={np.corrcoef(x1*x2,y_mini)[0,1]:.3f})')

plt.tight_layout(); plt.show()

# Visualizacao 2: R² com interacao real vs com ruido
labels_mini = list(r2_mini.keys())
vals_mini   = list(r2_mini.values())
colors_mini = [C_BASE, C_GOOD, C_BAD]

fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.bar(labels_mini, vals_mini, color=colors_mini, edgecolor='white', width=0.5)
ax.bar_label(bars, fmt='%.4f', padding=3, fontsize=11, fontweight='bold')
ax.set_ylim(0, 1.07)
ax.set_ylabel('R² (CV 5-fold)')
ax.set_title('Feature informativa vs ruido: o contraste no mini-dataset',
             fontweight='bold')
for i in range(1, len(vals_mini)):
    delta = vals_mini[i] - vals_mini[0]
    ax.annotate(f'{"+" if delta>=0 else ""}{delta:.4f}',
                xy=(i, vals_mini[i] + 0.02), ha='center', fontsize=10,
                color=C_GOOD if delta > 0 else C_BAD, fontweight='bold')
plt.tight_layout(); plt.show()

print('Mini-dataset (N=500, 5 folds):')
for lbl, val in r2_mini.items():
    print(f'  {lbl:<35}: R² = {val:.4f}')

# ── Parte 2: dataset de imoveis — informativo vs ruido ───────────────
print()
print('Dataset de imoveis (N=800):')
print('Nota: o efeito das interacoes e mais sutil aqui porque area ja e')
print('muito correlacionada com area*quartos — o Ridge suprime a redundancia.')
print()

X_int = build_interacoes(df, X_base)
rng_noise = np.random.default_rng(77)

sets_imoveis = {
    'Baseline':                  X_base,
    '+ 2 inter. de dominio':     X_int[X_base.columns.tolist() +
                                         ['area_x_quartos','area_x_dist']],
    '+ 4 inter. de dominio':     X_int,
    '+ 8 inter. de ruido':       pd.concat([
                                    X_base,
                                    pd.DataFrame({
                                        f'ruido_{k}': rng_noise.permutation(df['area'].values)
                                                   * rng_noise.permutation(df['quartos'].values)
                                        for k in range(8)
                                    })], axis=1),
}

r2_imoveis = {}
for label, X_d in sets_imoveis.items():
    pipe = Pipeline([('sc', StandardScaler()), ('r', Ridge(alpha=10))])
    r2s  = cross_val_score(pipe, X_d, y, cv=kf5, scoring='r2')
    r2_imoveis[label] = (r2s.mean(), r2s.std(), X_d.shape[1])

fig, ax = plt.subplots(figsize=(10, 4))
labels_im = list(r2_imoveis.keys())
vals_im   = [v[0] for v in r2_imoveis.values()]
nfeats_im = [v[2] for v in r2_imoveis.values()]
colors_im = [C_BASE, C_GOOD, C_GOOD, C_BAD]
bars = ax.bar(labels_im, vals_im, color=colors_im, edgecolor='white', width=0.5)
ax.bar_label(bars, fmt='%.4f', padding=3, fontsize=11, fontweight='bold')
for i in range(1, len(vals_im)):
    delta = vals_im[i] - vals_im[0]
    ax.annotate(f'{"+" if delta>=0 else ""}{delta:.4f}',
                xy=(i, vals_im[i] + 0.005), ha='center', fontsize=9,
                color=C_GOOD if delta > 0 else C_BAD)
ax.set_ylim(ax.get_ylim()[0]*0.99, ax.get_ylim()[1]*1.03)
ax.set_ylabel('R² (CV 5-fold)')
ax.set_title('Dataset de imoveis: interacoes de dominio vs ruido',
             fontweight='bold')
plt.tight_layout(); plt.show()

for lbl, (r2_val, std_val, nf) in r2_imoveis.items():
    print(f'  {lbl:<35}: R² = {r2_val:.4f}  ({nf} features)')

# Armazenar R² da configuracao com 4 interacoes de dominio para uso no proximo experimento
r2_int = r2_imoveis['+ 4 inter. de dominio'][0]


**Conclusão:**

O mini-dataset demonstra de forma inequívoca o papel de uma interação informativa:
sem `x1*x2`, o Ridge não consegue capturar o fenômeno porque `x1` e `x2`
individualmente são fracos preditores do alvo. Com a interação correta, R²
salta de 0,88 para 0,99. Colunas de ruído (permutações aleatórias) não movem
o agulha — o Ridge zera seus coeficientes.

No dataset de imóveis o efeito das interações é mais sutil por uma razão
estrutural: `area` é tão dominante (ρ = 0,77 com o alvo) que `area × quartos`
é quase colinear com `area`. O Ridge, ao regularizar, suprime o termo redundante.
Ainda assim, **o ruído sempre prejudica** — cada interação sem sinal real
adiciona uma direção espúria que o Ridge precisa suprimir, consumindo capacidade
de regularização que poderia ser usada nas features relevantes.

**Regra prática:** crie interações motivadas pelo negócio, não por exaustão.
Verificar correlação entre a interação candidata e o resíduo do modelo sem
ela é uma forma rápida de avaliar se há sinal real.


### 5.2 Polinômios

**Objetivo:** capturar relações curvas entre uma variável e o alvo,
especialmente a relação quadrática entre `area` e `preco` — embutida
intencionalmente no dataset.

**Racional:** o Ridge com apenas `area` aprende uma reta. Com `area²`
disponível, o modelo pode ajustar uma parábola — muito mais próxima da
relação real. O risco é que `PolynomialFeatures(degree=2)` aplicado a todas
as colunas gera *n(n+3)/2* features, crescimento que pode causar overfitting
mesmo com regularização.


In [ ]:
print('=' * 60)
print('POLINOMIOS — termos quadraticos e cubicos para relacoes curvas')
print('=' * 60)

X_poly = build_polinomios(df, X_int)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle('Por que area² importa?', fontweight='bold')

axes[0].scatter(df['area'], df['preco']/1e6, alpha=0.3, s=15, color=C_BASE)
axes[0].set_xlabel('Area (m2)'); axes[0].set_ylabel('Preco (R$ M)')
axes[0].set_title('Relacao curva: preco sobe mais que linearmente')

x_sorted = np.sort(df['area'].values)
axes[1].scatter(df['area'], df['preco']/1e6, alpha=0.25, s=15, color=C_BASE)
axes[1].plot(x_sorted, np.polyval(np.polyfit(df['area'], df['preco']/1e6, 1), x_sorted),
             color=C_BAD,  lw=2.5, label='Ajuste linear')
axes[1].plot(x_sorted, np.polyval(np.polyfit(df['area'], df['preco']/1e6, 2), x_sorted),
             color=C_GOOD, lw=2.5, label='Ajuste quadratico')
axes[1].set_xlabel('Area (m2)')
axes[1].set_title('Linear vs Quadratico — qual captura melhor?')
axes[1].legend()

plt.tight_layout(); plt.show()

# Comparacao: selecao manual vs PolynomialFeatures global
print('Comparacao: polinomios manuais vs PolynomialFeatures global:')
graus, r2s_poly, n_feats = [], [], []
for grau in [1, 2, 3]:
    pf    = PolynomialFeatures(degree=grau, include_bias=False)
    X_tmp = pf.fit_transform(X_base)
    r2    = avaliar(pd.DataFrame(X_tmp), y, f'PolyFeatures grau={grau}')
    graus.append(f'Grau {grau}\n({X_tmp.shape[1]} feat.)')
    r2s_poly.append(r2); n_feats.append(X_tmp.shape[1])

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(graus, r2s_poly, color=[C_BASE, C_GOOD, C_WARN], edgecolor='white', width=0.5)
ax.bar_label(bars, fmt='%.4f', padding=3, fontsize=11)
ax.set_ylim(0, 1); ax.set_ylabel('R² (CV 5-fold)')
ax.set_title('Grau polinomial global vs R²\nCuidado com a explosao de features!',
             fontweight='bold')
plt.tight_layout(); plt.show()

r2_poly = avaliar(X_poly, y, 'Polinomios manuais (4 features)')
print(f'  Interacoes (4 features):  R² = {r2_int:.4f}')
print(f'  + Polinomios (4 features): R² = {r2_poly:.4f}  (+{r2_poly-r2_int:.4f})')
print(f'  Novas colunas: {X_poly.columns[len(X_int.columns):].tolist()}')


**Conclusão:** termos polinomiais selecionados manualmente trazem ganho sem
o custo dimensional do `PolynomialFeatures` global. Aplicar grau 2 a todas
as features com Ridge pode até superar o modelo manual em R², mas produz
dezenas de colunas com pouco sinal individual — frágil em dados reais com
menos amostras ou mais ruído. Grau 3 global raramente compensa o custo.


### 5.3 Agregações e Data Leakage

**Objetivo:** mostrar o poder das estatísticas por grupo (o bairro como
contexto) e demonstrar de forma explícita como o data leakage surge e
contamina a avaliação.

**Racional:** o bairro carrega informação coletiva (infraestrutura, valorização
histórica) que nenhuma feature individual do imóvel captura. A média do
preço por bairro é uma feature poderosa — mas se calculada com os dados de
validação incluídos, cada amostra "vê" o seu próprio alvo, e o modelo
aprende o gabarito em vez de um padrão generalizável.


In [ ]:
print('=' * 60)
print('AGREGACOES — estatisticas por grupo + demonstracao de data leakage')
print('=' * 60)

# Estatisticas por bairro
bairro_stats = df.groupby('bairro')['preco'].agg(['mean','median','std','count']).round(0)
bairro_stats.columns = ['media', 'mediana', 'desvio', 'n']
print('Estatisticas de preco por bairro:')
print(bairro_stats.to_string())

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle('Agregacoes por Bairro', fontweight='bold')
order = bairro_stats.sort_values('media').index
axes[0].barh(order, bairro_stats.loc[order, 'media']/1e6,
             color=C_PURPLE, edgecolor='white')
axes[0].set_xlabel('Preco medio (R$ M)')
axes[0].set_title('Preco medio por bairro')
df.boxplot(column='preco', by='bairro', ax=axes[1], grid=False)
axes[1].set_title('Distribuicao de preco por bairro')
axes[1].set_xlabel(''); axes[1].set_ylabel('Preco (R$)')
plt.suptitle('')
plt.tight_layout(); plt.show()

# ── Por que o R² sobe tanto neste dataset? ──────────────────────────
# O dataset sintetico tem multiplicadores EXATOS por bairro (1.6, 1.1, etc.)
# O Target Encoding OOF essencialmente RECUPERA esses multiplicadores
# a partir de ~130 amostras por grupo — que e muito mais do que o necessario.
# Com ~130 amostras por bairro, a media e muito estavel e o encoding
# captura quase toda a variacao do fator bairro no preco.
#
# Em dados reais, a variancia dentro do grupo e muito maior (imoveis do
# mesmo bairro variam amplamente), entao o ganho seria menor.
# Aqui, o ganho de +0.20 reflete que o bairro e uma variavel latente
# que o modelo nao conseguia observar diretamente — e a agregacao
# a torna observavel.

print()
print('Nota sobre o ganho de R²:')
print('  O bairro tem um multiplicador EXATO no dataset sintetico.')
print('  Com ~130 amostras por bairro, o TE recupera esse multiplicador')
print('  com alta precisao. Em dados reais, o ganho seria menor porque')
print('  a variacao dentro do grupo e muito mais alta.')
print()

# ── Demonstracao de leakage: por que holdout nao basta ──────────────
# Com ~130 amostras por bairro, cada amostra representa apenas 0.8%
# da media do grupo. O leakage e imperceptivel no holdout.
#
# Para tornar o leakage claramente visivel, usamos validacao cruzada
# com poucos dados — a mesma abordagem do notebook de encoding.
# Mini-dataset: 10 bairros, 2 amostras cada (50% de contribuicao).

print('=== DEMONSTRACAO DE DATA LEAKAGE ===')
print('Por que o holdout nao revela leakage em bairros com muitas amostras:')
print(f'  Cada amostra representa ~{100/(len(df)/5):.1f}% da media do seu bairro.')
print('  Influencia tao pequena que o modelo com e sem leakage')
print('  tem R² praticamente igual no conjunto de teste.')
print()
print('Para tornar o leakage visivel: 10 bairros, 2 amostras cada (50% por amostra).')
print()

np.random.seed(SEED)
n_bairros_demo, n_per = 10, 2
bairros_demo   = [f'bairro_{i:02d}' for i in range(n_bairros_demo)]
bairro_effects = np.linspace(0, 10, n_bairros_demo)
b_col   = np.repeat(bairros_demo, n_per)
x_num_d = np.random.randn(n_bairros_demo * n_per)
y_demo  = pd.Series(
    np.array([bairro_effects[i] for i in range(n_bairros_demo) for _ in range(n_per)])
    + 0.2 * x_num_d
    + np.random.randn(n_bairros_demo * n_per) * 0.3
)
df_demo = pd.DataFrame({'bairro': b_col, 'x': x_num_d})

# Mapa ERRADO: media calculada no dataset completo (com todas as amostras)
te_global = y_demo.groupby(pd.Series(b_col)).mean()
X_errado  = pd.DataFrame({'x': x_num_d,
                           'te_bairro': pd.Series(b_col).map(te_global)})

# Mapa CORRETO: recalculado dentro de cada fold do CV
kf5 = KFold(n_splits=5, shuffle=True, random_state=SEED)
r2_wrong_list, r2_correct_list = [], []

for tr_idx, va_idx in kf5.split(df_demo):
    # ERRADO: usa o mapa calculado com TODAS as amostras (inclui validacao)
    Xw_tr = X_errado.iloc[tr_idx]
    Xw_va = X_errado.iloc[va_idx]
    sc_w = StandardScaler(); r_w = Ridge(alpha=1)
    r_w.fit(sc_w.fit_transform(Xw_tr), y_demo.iloc[tr_idx])
    r2_wrong_list.append(
        r2_score(y_demo.iloc[va_idx], r_w.predict(sc_w.transform(Xw_va)))
    )

    # CORRETO: media calculada SOMENTE com o fold de treino
    gm = y_demo.iloc[tr_idx].mean()
    te_fold = y_demo.iloc[tr_idx].groupby(
        pd.Series(b_col)[tr_idx].values
    ).mean()
    Xc_tr = pd.DataFrame({
        'x': x_num_d[tr_idx],
        'te_bairro': pd.Series(b_col)[tr_idx].map(te_fold).fillna(gm)
    })
    Xc_va = pd.DataFrame({
        'x': x_num_d[va_idx],
        'te_bairro': pd.Series(b_col)[va_idx].map(te_fold).fillna(gm)
    })
    sc_c = StandardScaler(); r_c = Ridge(alpha=1)
    r_c.fit(sc_c.fit_transform(Xc_tr), y_demo.iloc[tr_idx])
    r2_correct_list.append(
        r2_score(y_demo.iloc[va_idx], r_c.predict(sc_c.transform(Xc_va)))
    )

r2_leak_cv  = float(np.mean(r2_wrong_list))
r2_corr_cv  = float(np.mean(r2_correct_list))

fig, ax = plt.subplots(figsize=(8, 4))
labels_l  = ['ERRADO\n(TE antes do CV)\n50% contribuicao/amostra',
              'CORRETO\n(TE dentro do CV)\nOOF — sem leakage']
vals_l    = [r2_leak_cv, r2_corr_cv]
bars = ax.bar(labels_l, vals_l, color=[C_BAD, C_GOOD], edgecolor='white', width=0.4)
ax.bar_label(bars, fmt='%.4f', padding=3, fontsize=12, fontweight='bold')
ax.set_ylim(0, 1.1)
ax.set_ylabel('R² medio (CV 5-fold)')
ax.set_title(f'Leakage em Agregacoes\n'
             f'(10 bairros, {n_per} amostras cada — {100/n_per:.0f}% contribuicao por amostra)',
             fontweight='bold')
plt.tight_layout(); plt.show()

print(f'  CV R² ERRADO  (TE antes do CV):     {r2_leak_cv:.4f}  <- inflado pelo leakage')
print(f'  CV R² CORRETO (TE dentro do CV):    {r2_corr_cv:.4f}  <- estimativa honesta')
print(f'  Inflacao pelo leakage:              +{r2_leak_cv - r2_corr_cv:.4f}')

# Agregacao correta via OOF no dataset principal
y_series = pd.Series(y.values, index=df.index)
X_agg    = build_agregacoes(df, y_series, X_poly)
r2_agg   = avaliar(X_agg, y, 'Agregacoes (Target Encoding OOF)')
print()
print(f'  Polinomios:    R² = {r2_poly:.4f}')
print(f'  + Agregacoes:  R² = {r2_agg:.4f}  (+{r2_agg-r2_poly:.4f})')
print()
print(f'  Novas colunas: {X_agg.columns[len(X_poly.columns):].tolist()}')


**Conclusão:**

O salto de R² (+0,20) é real, mas reflete uma particularidade do dataset
sintético: cada bairro tem um multiplicador de preço **exato e estável**
(1,6 para Centro, 0,85 para Sul etc.). Com ~130 amostras por bairro, o
Target Encoding OOF recupera esse multiplicador com alta precisão — o
modelo passa a "ver" a variável latente que até então estava oculta.

Em dados reais, a variância dentro de cada bairro é muito maior: imóveis
no mesmo bairro diferem amplamente por microlocalizações, estado de
conservação e outros fatores. O ganho seria real mas mais modesto,
tipicamente +0,03 a +0,06 de R².

**Por que o holdout não revela leakage aqui:**  
Com ~130 amostras por bairro, cada amostra representa apenas ~0,8% da
média do grupo. A "contaminação" de uma única amostra de teste no cálculo
da média é imperceptível. O leakage só se torna visível quando há poucas
amostras por categoria — como demonstrado acima com 10 bairros e 2 amostras
cada (50% de contribuição por amostra), onde a inflação pelo leakage é de
mais de 0,23 pontos de R².

**Regra:** sempre use OOF ou recalcule as agregações dentro do loop de CV,
independentemente do tamanho do grupo — o problema se torna grave
exatamente quando os grupos são pequenos.


### 5.4 Transformações

**Objetivo:** corrigir distribuições assimétricas (*skew* alto) que tornam
o aprendizado do Ridge instável, e demonstrar que `log1p`, `sqrt` e Box-Cox
reduzem a assimetria de forma mensurável.

**Racional:** o Ridge minimiza o erro quadrático assumindo implicitamente que
os resíduos são aproximadamente normais. Variáveis com cauda longa (como `area`
e `dist_centro`) produzem resíduos grandes nos extremos, forçando o modelo a
sacrificar a qualidade de ajuste nas regiões centrais. Comprimindo as caudas
com transformações logarítmicas ou de raiz quadrada, os coeficientes se tornam
mais estáveis e a convergência melhora.


In [ ]:
print('=' * 60)
print('TRANSFORMACOES — log, sqrt, Box-Cox para corrigir distribuicoes')
print('=' * 60)

fig, axes = plt.subplots(2, 4, figsize=(16, 7))
fig.suptitle('Transformacoes: antes e depois', fontweight='bold', fontsize=13)

vars_transform = [
    ('area',        'log1p(area)',        lambda x: np.log1p(x)),
    ('dist_centro', 'sqrt(dist_centro)',  lambda x: np.sqrt(x)),
    ('preco',       'log1p(preco)',       lambda x: np.log1p(x)),
    ('idade',       'log1p(idade)',       lambda x: np.log1p(x)),
]

for i, (col, label, fn) in enumerate(vars_transform):
    vals = df[col].values
    sk_before = stats.skew(vals)
    transf = fn(vals)
    sk_after  = stats.skew(transf)
    axes[0, i].hist(vals,   bins=30, color=C_WARN, edgecolor='white', alpha=0.85)
    axes[0, i].set_title(f'{col}  (skew={sk_before:.2f})', fontsize=9)
    axes[1, i].hist(transf, bins=30, color=C_GOOD, edgecolor='white', alpha=0.85)
    axes[1, i].set_title(f'{label}  (skew={sk_after:.2f})', fontsize=9)

axes[0, 0].set_ylabel('Antes',  fontweight='bold', fontsize=11)
axes[1, 0].set_ylabel('Depois', fontweight='bold', fontsize=11)
plt.tight_layout(); plt.show()

X_transf, lmbda = build_transformacoes(df, X_agg)
print(f'Box-Cox lambda para area: {lmbda:.3f}')
print('  (lambda=0 equivale a log | lambda=0.5 equivale a sqrt)')

r2_transf = avaliar(X_transf, y, 'Transformacoes (log, sqrt, boxcox)')
print(f'  Agregacoes:       R² = {r2_agg:.4f}')
print(f'  + Transformacoes: R² = {r2_transf:.4f}  (+{r2_transf-r2_agg:.4f})')
print(f'  Novas colunas: {X_transf.columns[len(X_agg.columns):].tolist()}')


**Conclusão:** transformações reduzem a assimetria das variáveis de forma
mensurável (o *skew* cai substancialmente). O ganho de R² tende a ser
incremental quando as interações e polinômios já capturam a não-linearidade
principal — mas as transformações melhoram a estabilidade dos coeficientes e
reduzem a sensibilidade a *outliers*. Para dados reais com distribuições mais
extremas, o ganho costuma ser maior.


### 5.5 Domínio Específico

**Objetivo:** demonstrar que features construídas com conhecimento do
negócio frequentemente superam combinações matemáticas genéricas, porque
codificam regras que especialistas levam anos para aprender.

**Racional:** um corretor experiente sabe que "área por quarto" mede conforto
de uma forma que `area` e `quartos` separados não capturam. A "flag premium"
(distância < 5 km) não é inferida automaticamente de `dist_centro` por um
modelo linear — exige o limiar explícito que o corretor conhece. Essas
features traduzem regras do domínio em sinal preditivo direto.


In [ ]:
print('=' * 60)
print('DOMINIO ESPECIFICO — conhecimento do mercado imobiliario')
print('=' * 60)

X_domain    = build_dominio(df, X_transf)
domain_cols = [c for c in X_domain.columns if c not in X_transf.columns]
print(f'Features de dominio criadas ({len(domain_cols)}): {domain_cols}')

# Correlacao das features de dominio com o target
corr_data = X_domain[domain_cols].copy()
corr_data['log_preco'] = y.values
corrs  = corr_data.corr()['log_preco'].drop('log_preco').sort_values()
colors = [C_GOOD if v > 0 else C_BAD for v in corrs]

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
fig.suptitle('Features de dominio: correlacao e distribuicao', fontweight='bold')

axes[0].barh(corrs.index, corrs.values, color=colors, edgecolor='white')
axes[0].axvline(0, color='gray', lw=0.8)
axes[0].set_xlabel('Correlacao de Pearson com log(preco)')
axes[0].set_title('Correlacao das features de dominio com o alvo')

# Boxplot: preco por flag premium
df['is_premium_cat'] = (df['dist_centro'] < 5).map({True: 'Premium (<5km)', False: 'Padrao (>=5km)'})
df.boxplot(column='preco', by='is_premium_cat', ax=axes[1], grid=False)
axes[1].set_title('Preco por categoria de localizacao')
axes[1].set_xlabel(''); axes[1].set_ylabel('Preco (R$)')
plt.suptitle('')
plt.tight_layout(); plt.show()
df.drop(columns=['is_premium_cat'], inplace=True)

r2_domain = avaliar(X_domain, y, 'Dominio especifico (7 features)')
print(f'  Transformacoes:    R² = {r2_transf:.4f}')
print(f'  + Dominio:         R² = {r2_domain:.4f}  (+{r2_domain-r2_transf:.4f})')


**Conclusão:** features de domínio têm o maior retorno por feature adicionada
neste experimento. A flag `is_premium` sozinha separa dois regimes de preço
completamente distintos — algo que o modelo linear não consegue inferir apenas
de `dist_centro`. O índice de comodidade (`quartos + banheiros × 0,8 + vagas × 0,5`)
comprime três features em um único score que o Ridge consegue usar com um
único coeficiente estável.


### 5.6 Scaling

**Objetivo:** demonstrar o impacto do escalonamento sobre o Ridge e comparar
`StandardScaler`, `MinMaxScaler` e `RobustScaler` em presença de *outliers*.

**Racional:** o Ridge minimiza `||y - Xβ||² + α||β||²`. Sem escalonamento,
features com magnitudes muito diferentes (ex.: `area` em centenas vs
`is_premium` em {0,1}) forçam o modelo a aprender coeficientes em escalas
incompatíveis, prejudicando tanto o ajuste quanto a interpretação da
regularização. O `RobustScaler` é especialmente relevante para dados reais
com *outliers* legítimos (imóveis de altíssimo padrão, por exemplo).


In [ ]:
print('=' * 60)
print('SCALING — comparativo de StandardScaler, MinMaxScaler e RobustScaler')
print('=' * 60)

# Introduzir outliers artificiais em area para demonstrar RobustScaler
df_out = df.copy()
idx_out = df_out.sample(15, random_state=SEED).index
df_out.loc[idx_out, 'area'] = 1200   # imoveis com area extrema
X_out = df_out[BASE_COLS].copy()

# Visualizar histogramas apos cada scaler
feature = df_out['area'].values
scalers_demo = [
    ('Original',       feature),
    ('StandardScaler', StandardScaler().fit_transform(feature.reshape(-1,1)).ravel()),
    ('MinMaxScaler',   MinMaxScaler().fit_transform(feature.reshape(-1,1)).ravel()),
    ('RobustScaler',   RobustScaler().fit_transform(feature.reshape(-1,1)).ravel()),
]

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
fig.suptitle('Scalers: comportamento com outliers de area (15 imoveis artificiais)',
             fontweight='bold')
colors_sc = [C_GRAY, C_BASE, C_WARN, C_GOOD]
for ax, (name, data), color in zip(axes, scalers_demo, colors_sc):
    p5, p95 = np.percentile(data, [5, 95])
    ax.hist(data, bins=25, color=color, alpha=0.85, edgecolor='white')
    ax.axvline(p5,  color='red', lw=1.2, ls='--', alpha=0.7, label='P5/P95')
    ax.axvline(p95, color='red', lw=1.2, ls='--', alpha=0.7)
    ax.set_title(f'{name}\nskew={abs(pd.Series(data).skew()):.2f}', fontweight='bold')
    ax.set_xlabel('Valor')
    if ax == axes[0]:
        ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

# Impacto no R² com e sem outliers
print('Impacto do scaler no Ridge (dataset original, sem outliers artificiais):')
resultados_sc = {}
for sc_name, sc in [
    ('StandardScaler', StandardScaler()),
    ('MinMaxScaler',   MinMaxScaler()),
    ('RobustScaler',   RobustScaler()),
    ('Sem scaling',    None),
]:
    X_sc = X_base.copy()
    if sc is not None:
        X_sc = pd.DataFrame(sc.fit_transform(X_sc), columns=X_sc.columns)
    cv   = KFold(n_splits=5, shuffle=True, random_state=SEED)
    r2s  = cross_val_score(Ridge(alpha=10), X_sc, y, cv=cv, scoring='r2')
    resultados_sc[sc_name] = r2s.mean()
    print(f'  {sc_name:<20}: R² = {r2s.mean():.4f}')

print()
print('Com outliers artificiais em area:')
for sc_name, sc in [
    ('StandardScaler', StandardScaler()),
    ('RobustScaler',   RobustScaler()),
]:
    X_sc = pd.DataFrame(sc.fit_transform(X_out), columns=X_out.columns)
    cv   = KFold(n_splits=5, shuffle=True, random_state=SEED)
    r2s  = cross_val_score(Ridge(alpha=10), X_sc, y, cv=cv, scoring='r2')
    print(f'  {sc_name:<20}: R² = {r2s.mean():.4f}  (com outliers artificiais)')


**Conclusão:** com dados limpos, os três scalers produzem resultados
semelhantes — a diferença entre eles é pequena quando não há *outliers*.
A distinção aparece com dados que possuem valores extremos legítimos: o
`StandardScaler` tem sua média e desvio distorcidos pelos *outliers*,
comprimindo todos os dados normais em uma faixa estreita. O `RobustScaler`,
baseado em mediana e IQR, mantém a escala estável. Para dados imobiliários
reais (imóveis de altíssimo padrão existem e são legítimos), `RobustScaler`
é a escolha mais segura.


### 5.7 Feature Learning com PCA

**Objetivo:** demonstrar como a Análise de Componentes Principais (PCA)
pode ser usada como extrator de *features* — criando representações
comprimidas que capturam a variância máxima dos dados originais.

**Racional:** PCA é o exemplo mais acessível de *feature learning* supervisionado
sem rótulos. Diferente das técnicas anteriores (que criavam features com base
em conhecimento humano), o PCA descobre automaticamente as direções de maior
variância no espaço de features. Em problemas reais com dezenas de variáveis
altamente correlacionadas (ex.: dados de sensores, pixels de imagem), os
componentes principais podem resumir a informação em muito menos dimensões
sem perda significativa de sinal.


In [ ]:
print('=' * 60)
print('FEATURE LEARNING — PCA como extrator de componentes latentes')
print('=' * 60)

# Aplicar PCA sobre o conjunto completo de features de dominio
sc_pca = StandardScaler()
X_scaled_pca = sc_pca.fit_transform(X_domain)

# Variancia explicada por numero de componentes
pca_full = PCA().fit(X_scaled_pca)
var_exp  = np.cumsum(pca_full.explained_variance_ratio_)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle('PCA como extrator de features', fontweight='bold')

# Painel 1: variancia acumulada
n_comp_95 = np.searchsorted(var_exp, 0.95) + 1
axes[0].plot(range(1, len(var_exp)+1), var_exp, 'o-', color=C_BASE, lw=2, ms=5)
axes[0].axhline(0.95, color=C_BAD, ls='--', label='95% variancia')
axes[0].axvline(n_comp_95, color=C_GOOD, ls='--',
                label=f'{n_comp_95} componentes para 95%')
axes[0].set_xlabel('Numero de componentes')
axes[0].set_ylabel('Variancia explicada acumulada')
axes[0].set_title('Variancia explicada pelo PCA')
axes[0].legend(fontsize=8); axes[0].grid(alpha=0.3)

# Painel 2: primeiros dois componentes coloridos por preco
pca2 = PCA(n_components=2, random_state=SEED).fit_transform(X_scaled_pca)
sc2  = axes[1].scatter(pca2[:, 0], pca2[:, 1], c=y, cmap='RdYlGn', alpha=0.6, s=20)
plt.colorbar(sc2, ax=axes[1], label='log(preco)')
axes[1].set_xlabel('PC1'); axes[1].set_ylabel('PC2')
axes[1].set_title('PC1 vs PC2 — cor = log(preco)')

# Painel 3: R² vs numero de componentes PCA
n_feats_total = X_domain.shape[1]
n_vals = list(range(1, min(n_feats_total+1, 21)))
r2_pca_list = []
cv5 = KFold(n_splits=5, shuffle=True, random_state=SEED)
for n in n_vals:
    pca_n  = PCA(n_components=n, random_state=SEED).fit_transform(X_scaled_pca)
    r2s    = cross_val_score(Ridge(alpha=10), pca_n, y, cv=cv5, scoring='r2')
    r2_pca_list.append(r2s.mean())

axes[2].plot(n_vals, r2_pca_list, 'o-', color=C_TEAL, lw=2, ms=6)
axes[2].axhline(avaliar(X_domain, y, 'Dominio (referencia PCA)'),
                color=C_WARN, ls='--', label='Todas as features originais')
axes[2].set_xlabel('Numero de componentes PCA')
axes[2].set_ylabel('R² (CV 5-fold)')
axes[2].set_title('R² com PCA vs numero de componentes', fontweight='bold')
axes[2].legend(fontsize=8); axes[2].grid(alpha=0.3)

plt.tight_layout(); plt.show()

# Avaliacao com PCA otimizado
best_n = n_vals[np.argmax(r2_pca_list)]
r2_pca_best = max(r2_pca_list)
RESULTADOS['Feature Learning (PCA otimo)'] = {
    'r2': r2_pca_best, 'std': 0.0, 'n_features': best_n
}
print(f'Numero de componentes que maximiza R²: {best_n} (de {n_feats_total})')
print(f'R² com {best_n} componentes PCA: {r2_pca_best:.4f}')
print(f'R² com todas as {n_feats_total} features originais: {RESULTADOS["Dominio (referencia PCA)"]["r2"]:.4f}')
print()
print(f'{n_comp_95} componentes explicam 95% da variancia do conjunto de features.')
print('O PCA encontra representacoes comprimidas que preservam o sinal preditivo.')


**Conclusão:** o PCA consegue representar a maior parte do sinal preditivo em
poucos componentes — geralmente menos do que o número original de features.
Isso demonstra que as features engenheiradas nas etapas anteriores são
fortemente correlacionadas entre si (redundância esperada), e que existe
estrutura latente capturável de forma não-supervisionada.

Em dados tabulares com poucos atributos (como este dataset), o PCA
raramente supera o uso direto das features originais. Seu valor real
aparece em alta dimensão: dados de sensores com centenas de canais,
imagens, embeddings de texto — onde compressão sem perda de sinal é
essencial para viabilizar o treinamento.


## 6. Resultados Consolidados

Todos os experimentos utilizam o mesmo protocolo: Ridge + StandardScaler +
validação cruzada com 5 partições. Os valores de R² são diretamente comparáveis.


In [ ]:
# Gráfico acumulativo: impacto de cada técnica
etapas = [
    ('Baseline',         'Baseline',                        C_BASE),
    ('+ Interacoes',     'Interacoes (4 features)',          C_PURPLE),
    ('+ Polinomios',     'Polinomios manuais (4 features)', C_PURPLE),
    ('+ Agregacoes',     'Agregacoes (Target Encoding OOF)', C_GOOD),
    ('+ Transformacoes', 'Transformacoes (log, sqrt, boxcox)', C_GOOD),
    ('+ Dominio',        'Dominio especifico (7 features)',   C_WARN),
]

labels_f, values_f, colors_f = [], [], []
for label_bar, label_res, cor in etapas:
    if label_res in RESULTADOS:
        labels_f.append(label_bar)
        values_f.append(RESULTADOS[label_res]['r2'])
        colors_f.append(cor)

fig, ax = plt.subplots(figsize=(13, 5))
bars = ax.bar(labels_f, values_f, color=colors_f, edgecolor='white', width=0.62)
ax.bar_label(bars, fmt='%.4f', padding=4, fontsize=10.5, fontweight='bold')

for i in range(1, len(values_f)):
    delta = values_f[i] - values_f[i-1]
    ax.annotate(f'{"+" if delta >= 0 else ""}{delta:.4f}',
                xy=(i, values_f[i] + 0.013), ha='center', fontsize=9,
                color=C_GOOD if delta > 0 else C_BAD)

ax.set_ylim(0, 1.05)
ax.set_ylabel('R² medio (CV 5-fold)', fontsize=11)
ax.set_title('Impacto acumulativo de cada tecnica de Feature Engineering\n'
             '(Ridge Regression + 5-fold CV)', fontsize=12, fontweight='bold')
legend = [
    mpatches.Patch(color=C_BASE,   label='Baseline'),
    mpatches.Patch(color=C_PURPLE, label='Complexidade estrutural'),
    mpatches.Patch(color=C_GOOD,   label='Contexto e escala'),
    mpatches.Patch(color=C_WARN,   label='Conhecimento de negocio'),
]
ax.legend(handles=legend, loc='lower right', fontsize=9)
plt.tight_layout(); plt.show()


In [ ]:
# Tabela de resultados
df_res = pd.DataFrame([
    {'Experimento': k,
     'R² medio': round(v['r2'], 4),
     '+/- desvio': round(v['std'], 4),
     'N features': v['n_features']}
    for k, v in RESULTADOS.items()
    if v['std'] > 0  # remove entradas sem CV real
]).sort_values('R² medio', ascending=False).reset_index(drop=True)

pd.set_option('display.max_colwidth', 50)
df_res


In [ ]:
# Importancia das features no modelo final (X_domain)
sc_final = StandardScaler()
X_sc_fin = sc_final.fit_transform(X_domain)
model_f  = RidgeCV(alphas=[0.1, 1, 10, 100]).fit(X_sc_fin, y)
print(f'Ridge alpha selecionado pelo RidgeCV: {model_f.alpha_}')

importances = (pd.Series(np.abs(model_f.coef_), index=X_domain.columns)
               .sort_values(ascending=True).tail(20))

def get_color(name):
    if any(k in name for k in ['_x_', 'x_']):     return C_PURPLE
    if any(k in name for k in ['2','3','boxcox']): return C_BASE
    if any(k in name for k in ['te_','media_','count_']): return C_GOOD
    if any(k in name for k in ['log_','sqrt_']):   return C_WARN
    return C_BAD

fig, ax = plt.subplots(figsize=(9, 7))
ax.barh(importances.index, importances.values,
        color=[get_color(n) for n in importances.index], edgecolor='white')
ax.set_xlabel('|Coeficiente Ridge| (features padronizadas)', fontsize=10)
ax.set_title('Top 20 features mais importantes no modelo final', fontweight='bold')
legend = [
    mpatches.Patch(color=C_PURPLE, label='Interacoes'),
    mpatches.Patch(color=C_BASE,   label='Polinomios'),
    mpatches.Patch(color=C_GOOD,   label='Agregacoes'),
    mpatches.Patch(color=C_WARN,   label='Transformacoes'),
    mpatches.Patch(color=C_BAD,    label='Originais / dominio'),
]
ax.legend(handles=legend, fontsize=9)
plt.tight_layout(); plt.show()


## 7. Conclusão e Regras Práticas

### O que os experimentos revelaram

| Técnica | Δ R² típico | Principal aprendizado |
|---------|-------------|----------------------|
| Baseline | — | Features brutas já capturam parte relevante da variância |
| Interações | +0,01–0,03 | Pares motivados pelo domínio têm sinal real; força bruta tem retorno decrescente |
| Polinômios | +0,01–0,02 | Seleção manual supera `PolynomialFeatures` global em relação sinal/custo |
| Agregações | **+0,03–0,06** | Contexto de grupo é muito poderoso — OOF é obrigatório para evitar leakage |
| Transformações | +0,01–0,02 | Corrige distribuições assimétricas; ganho maior em dados com *outliers* extremos |
| Domínio | **+0,02–0,04** | Maior retorno por feature adicionada; conhecimento do negócio é insubstituível |
| Scaling | Neutro–+0,01 | Essencial para Ridge; `RobustScaler` é mais seguro com *outliers* legítimos |
| PCA | Neutro–+0,01 | Útil em alta dimensão; em dados tabulares simples, raramente supera as features originais |

### Regras práticas

1. **Meça o baseline antes de qualquer transformação** — você precisa de um
   ponto de referência honesto.

2. **Valide cada adição com validação cruzada** — um único holdout é
   insuficiente para comparar técnicas; o CV 5-fold estabiliza a estimativa.

3. **Interações por domínio, não por exaustão** — gerar todas as combinações
   possíveis traz retorno decrescente e aumenta o risco de overfitting.

4. **Agregações exigem OOF obrigatoriamente** — calcular a média do grupo
   antes do CV é data leakage. A diferença de R² pode ser de 5–10 pontos
   percentuais em casos extremos.

5. **Transformações corrigem escala, não erros de modelagem** — se o modelo
   linear não captura o fenômeno, transformações ajudam, mas não substituem
   a engenharia de features correta.

6. **Domínio específico tem o maior retorno por feature** — quando há
   especialistas disponíveis, invista tempo entrevistando-os antes de qualquer
   automação.

7. **Scaling é obrigatório para Ridge, SVM e KNN; irrelevante para árvores** —
   use `RobustScaler` quando houver *outliers* legítimos no problema.

8. **Feature Learning (PCA, autoencoders) vale a pena em alta dimensão** —
   em dados tabulares com dezenas de features, o ganho raramente compensa a
   complexidade adicionada.
